In [1]:
import pandas as pd

reg = pd.read_csv("data/csv/raw/regimenes.csv", parse_dates=["date"], index_col="date")

print("Columnas:", reg.columns.tolist())
print(f"Filas: {len(reg)}  ({reg.index.min().date()} -> {reg.index.max().date()})")
print()

# Reparto de regímenes (debería ser ~Acumulación 39% / Bajista 36% / Alcista 25%)
col = "regimen" if "regimen" in reg.columns else "estado_hmm"
print("Reparto de regímenes:")
print(reg[col].value_counts())
print()
print("En porcentaje:")
print((reg[col].value_counts(normalize=True) * 100).round(1))
print()

# Cuántos cambios de régimen hay (deberían ser ~27-28)
cambios = (reg[col] != reg[col].shift()).sum()
print(f"Número de tramos (cambios de régimen): {cambios}")
print()

# El régimen de los últimos días
print("Últimos 5 días:")
print(reg[[col]].tail())

Columnas: ['precio', 'vol_30d', 'cum_ret_60d', 'dist_sma200', 'drawdown', 'fg', 'estado_hmm', 'regimen']
Filas: 2847  (2018-08-19 -> 2026-06-04)

Reparto de regímenes:
regimen
Acumulacion    1116
Bajista        1029
Alcista         702
Name: count, dtype: int64

En porcentaje:
regimen
Acumulacion    39.2
Bajista        36.1
Alcista        24.7
Name: proportion, dtype: float64

Número de tramos (cambios de régimen): 28

Últimos 5 días:
                regimen
date                   
2026-05-31  Acumulacion
2026-06-01  Acumulacion
2026-06-02  Acumulacion
2026-06-03  Acumulacion
2026-06-04  Acumulacion


In [1]:
import torch
ckpt = torch.load("models/lstm_final.pt", map_location="cpu", weights_only=False)
print("CLAVES:", list(ckpt.keys()))
print("arquitectura:", ckpt.get("arquitectura"))
print("seq_len:", ckpt.get("seq_len"), "| horizon:", ckpt.get("horizon"))
print("n feature_cols:", len(ckpt.get("feature_cols", [])))
print("n regime_cols:", len(ckpt.get("regime_cols", [])))
# y las formas reales de los pesos:
for k, v in ckpt["state_dict"].items():
    print(k, tuple(v.shape))

CLAVES: ['state_dict', 'arquitectura', 'feature_cols', 'regime_cols', 'seq_len', 'horizon', 'target_col']
arquitectura: {'n_features': 19, 'hidden_sizes': (48, 24), 'horizon': 3, 'dropout': 0.35}
seq_len: 30 | horizon: 3
n feature_cols: 16
n regime_cols: 3
lstms.0.weight_ih_l0 (128, 19)
lstms.0.weight_hh_l0 (128, 32)
lstms.0.bias_ih_l0 (128,)
lstms.0.bias_hh_l0 (128,)
head.0.weight (16, 32)
head.0.bias (16,)
head.3.weight (3, 16)
head.3.bias (3,)


In [2]:
import torch

# 1. Cargar el checkpoint actual
ckpt = torch.load("models/lstm_final.pt", map_location="cpu", weights_only=False)

# 2. Deducir la arquitectura REAL desde los pesos (la verdad)
state = ckpt["state_dict"]
n_features = state["lstms.0.weight_ih_l0"].shape[1]
n_capas = sum(1 for k in state if k.endswith(".weight_ih_l0"))
hidden_sizes = tuple(state[f"lstms.{i}.weight_ih_l0"].shape[0] // 4 for i in range(n_capas))
horizon = state["head.3.weight"].shape[0]

print("ANTES (corrupto):", ckpt["arquitectura"])

# 3. Corregir el campo arquitectura
ckpt["arquitectura"] = {
    "n_features": n_features,
    "hidden_sizes": hidden_sizes,
    "horizon": horizon,
    "dropout": ckpt["arquitectura"].get("dropout", 0.35),  # el dropout no afecta a la forma
}

print("DESPUÉS (correcto):", ckpt["arquitectura"])

# 4. Guardar (hago una copia de seguridad antes, por si acaso)
import shutil
shutil.copy("models/lstm_final.pt", "models/lstm_final_backup.pt")
torch.save(ckpt, "models/lstm_final.pt")
print("✓ Guardado. Backup en lstm_final_backup.pt")

ANTES (corrupto): {'n_features': 19, 'hidden_sizes': (48, 24), 'horizon': 3, 'dropout': 0.35}
DESPUÉS (correcto): {'n_features': 19, 'hidden_sizes': (32,), 'horizon': 3, 'dropout': 0.35}
✓ Guardado. Backup en lstm_final_backup.pt


In [5]:
import torch

# Si la celda no está en la raíz del repo, pon la ruta completa, p. ej.:
# RUTA = r"c:\Users\josit\CUARTO CURSO\TFG\TFG_ETH_FORECAST\models\lstm_final.pt"
RUTA = r"models/lstm_final.pt"

print(f"Cargando: {RUTA}\n")
try:
    ckpt = torch.load(RUTA, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(RUTA, map_location="cpu")

# 1) Qué hay dentro
print("Tipo del contenido:", type(ckpt).__name__)
if isinstance(ckpt, dict):
    print("\nClaves de primer nivel:")
    for k, v in ckpt.items():
        if isinstance(v, (list, tuple)):
            extra = f"({len(v)} elementos)"
        elif isinstance(v, dict):
            extra = f"({len(v)} claves)"
        elif hasattr(v, "shape"):
            extra = f"(shape {tuple(v.shape)})"
        else:
            extra = ""
        print(f"  - {k!r}: {type(v).__name__} {extra}")

# 2) Buscar columnas por nombres habituales
posibles = ["columnas", "columns", "cols", "features", "feature_cols",
            "feature_names", "input_cols", "x_cols", "variables", "feats"]
if isinstance(ckpt, dict):
    for clave in posibles:
        if clave in ckpt:
            cols = list(ckpt[clave])
            print(f"\n>>> Columnas en {clave!r} ({len(cols)}):")
            for i, c in enumerate(cols):
                print(f"   {i:>2}. {c}")

# 3) Por si acaso: rastrear cualquier lista de texto (candidata a columnas)
def buscar(obj, camino=""):
    out = []
    if isinstance(obj, (list, tuple)) and obj and all(isinstance(x, str) for x in obj):
        out.append((camino or "(raíz)", list(obj)))
    elif isinstance(obj, dict):
        for k, v in obj.items():
            out += buscar(v, f"{camino}[{k!r}]" if camino else f"[{k!r}]")
    return out

for camino, lista in buscar(ckpt):
    print(f"\nLista de texto en {camino} -> {len(lista)} elementos:")
    print("   " + ", ".join(lista))

Cargando: models/lstm_final.pt

Tipo del contenido: dict

Claves de primer nivel:
  - 'state_dict': OrderedDict (8 claves)
  - 'arquitectura': dict (4 claves)
  - 'feature_cols': list (16 elementos)
  - 'regime_cols': list (3 elementos)
  - 'seq_len': int 
  - 'horizon': int 
  - 'target_col': str 

>>> Columnas en 'feature_cols' (16):
    0. eth_cum_ret_30d
    1. btc_dominance_chg14d
    2. inflation_chg30
    3. eth_bb_width
    4. eth_mfi14
    5. eth_dist_sma200
    6. alt_dominance_diff
    7. eth_rsi14
    8. eth_vol_14d
    9. eth_mcap_ret
   10. eth_stoch_d
   11. n_miedo_ext_30d
   12. n_codicia_15d
   13. presion_ext_neta_15d
   14. fear_greed_scaled
   15. eth_close_ret

Lista de texto en ['feature_cols'] -> 16 elementos:
   eth_cum_ret_30d, btc_dominance_chg14d, inflation_chg30, eth_bb_width, eth_mfi14, eth_dist_sma200, alt_dominance_diff, eth_rsi14, eth_vol_14d, eth_mcap_ret, eth_stoch_d, n_miedo_ext_30d, n_codicia_15d, presion_ext_neta_15d, fear_greed_scaled, eth_close_r